# 05 – Forecast ranges (Milestone 4, step 1)

**Project:** Perishable Inventory Optimizer · **Product:** bread and bakery

## What we are trying to accomplish
Turn each single-number forecast into a **range** (for example "between 2,600 and 3,100 units, 80% of the time"), so the stocking rule in the next step can weigh the cost of running out against the cost of waste.

## What we got accomplished
We turned each one-number forecast into a **range**, using how far off the model was on days it had never trained on. We tested four ways of grouping those past errors, picked the simplest one that held up (**one pooled set of ratios for all warehouses**), and scored it once on the 56 test days. The 80% range contained **90.6%** of the test days (target 80%); the 90% range contained 96.9%. Saved for the stocking rule: `forecast_ranges_test.csv` and `ratio_percentiles.csv`.

## What we have left to do
- **Milestone 4, step 2: the stocking rule.** Choose how much stock to hold from the range, weighing the cost of running out against the cost of waste. The data has no prices or margins, so we need to agree on an assumed cost ratio first.
- **Milestone 5:** the Streamlit app that shows the recommendation per warehouse.

---
### Method in one paragraph (what we are testing)
For each past day we know the forecast and what actually sold. The **ratio** actual ÷ forecast tells us how far off the model tends to be, as a percentage rather than in units, so a big warehouse and a small one can be compared. We take the validation days (the model never trained on them), sort the ratios, and read off percentiles. Multiply a new forecast by the 10th and 90th percentile ratios and you have an 80% range. We then check on the **test** days whether about 80% of real sales really land inside.

## 5a – Load the forecasts and compute the ratio
*What we are looking for:* one row per warehouse-day with the model's forecast, the actual, and the ratio between them.

In [10]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

PROCESSED = Path("../data/processed")
res = pd.read_csv(PROCESSED / "results_c.csv", parse_dates=["date"])
res = res[res["model"] == "c_hgb_ratio"].copy()
res["ratio"] = res["actual"] / res["forecast"]
res["weekday"] = res["date"].dt.day_name()

val = res[res["split"] == "validation"].copy()
test = res[res["split"] == "test"].copy()
print(len(val), "validation rows,", len(test), "test rows")
print(val["ratio"].describe().round(3))
print()
print("Validation ratio percentiles:")
print(val["ratio"].quantile([.05, .10, .25, .5, .75, .90, .95]).round(3))

224 validation rows, 224 test rows
count    224.000
mean       1.001
std        0.078
min        0.775
25%        0.957
50%        0.998
75%        1.051
max        1.295
Name: ratio, dtype: float64

Validation ratio percentiles:
0.05    0.873
0.10    0.899
0.25    0.957
0.50    0.998
0.75    1.051
0.90    1.095
0.95    1.112
Name: ratio, dtype: float64


### How we got here (the process)
We kept only the chosen model (`c_hgb_ratio`) from `results_c.csv`. Its validation forecasts come from a model trained on the training years only, so they are honest errors. The **ratio** (actual ÷ forecast) is used instead of units, so a big warehouse and a small one can be compared.

### How to read it
A ratio of 1.00 means a perfect forecast. 1.10 means we sold 10% *more* than forecast (a stockout risk). 0.90 means we sold 10% *less* (a waste risk).

### What it means
The ratios are centered on 1.00 (median 0.998, mean 1.001), so the model is not leaning high or low. The middle half of days is within about 4–5% of the forecast (0.957 to 1.051). The 10th and 90th percentiles are **0.899 and 1.095**, so an 80% range is roughly the forecast minus 10% to plus 10%. Example: a forecast of 3,000 units becomes about **2,700 to 3,285**.

### What's next and why
The percentiles pool all four warehouses and all weekdays. The next step checks whether splitting them up gives better ranges.

## 5b – Which grouping should the ranges use?
*What we are looking for:* a grouping that is detailed enough to be fair to each warehouse but not so detailed that each group has too few days. Options: one pooled set of ratios, per warehouse, per weekday, per warehouse×weekday.

*How we test:* calibrate on the **first 28** validation days, check how often the **last 28** fall inside the 80% range. Target is 80%.

In [11]:
days = sorted(val["date"].unique())
first, last = days[:28], days[28:]
cal = val[val["date"].isin(first)]
chk = val[val["date"].isin(last)]

LO, HI = 0.10, 0.90   # an 80% range

def coverage_and_width(calib, check, by):
    """Learn ratio percentiles on calib (per group), apply to check."""
    if by:
        q = calib.groupby(by)["ratio"].quantile([LO, HI]).unstack()
        q.columns = ["lo", "hi"]
        d = check.merge(q.reset_index(), on=by, how="left")
    else:
        d = check.copy()
        d["lo"], d["hi"] = calib["ratio"].quantile(LO), calib["ratio"].quantile(HI)
    lo, hi = d["forecast"] * d["lo"], d["forecast"] * d["hi"]
    inside = ((d["actual"] >= lo) & (d["actual"] <= hi)).mean()
    width = ((hi - lo) / d["forecast"]).mean()
    return inside, width

options = {
    "pooled": None,
    "by warehouse": ["warehouse"],
    "by weekday": ["weekday"],
    "warehouse x weekday": ["warehouse", "weekday"],
}
rows = []
for name, by in options.items():
    inside, width = coverage_and_width(cal, chk, by)
    rows.append({"grouping": name, "inside the range": round(inside, 3),
                 "avg width (% of forecast)": round(width, 3)})
pd.DataFrame(rows)

,grouping,inside the range,avg width (% of forecast)
0,pooled,0.750,0.182
1,by warehouse,0.786,0.180
2,by weekday,0.714,0.174
3,warehouse x weekday,0.571,0.128


### How we got here (the process)
Options tried: one pooled set of ratios; one set per warehouse; one per weekday; one per warehouse and weekday. We learned each on the first 28 validation days and checked the last 28, because choosing on the test days would use up the test.

### How to read it
"Inside the range" should be near **0.80**. "Width" is how wide the range is as a share of the forecast. Narrower is more useful, as long as coverage holds.

### What it means
Pooled 0.750 and by warehouse 0.786 are close to 0.80. By weekday is 0.714. Warehouse × weekday collapses to 0.571, because each group has only 8 days of ratios, so its percentiles are unreliable. With 112 days checked, a gap of a few points between pooled and by-warehouse is within chance, so this check cannot separate them.

### What's next and why
We drop warehouse × weekday and by weekday. Pooled has the most data behind each percentile (224 ratios), so we prefer it unless the next check shows a clear reason to split by warehouse.

## 5c – Is the range wide enough on its own?
*What we are looking for:* the share of days inside the range for several range sizes (50%, 80%, 90%). A well-calibrated range sits on the diagonal: ask for 80%, get about 80%.

In [12]:
LEVELS = [0.5, 0.8, 0.9]

def interval(calib, check, by, level):
    a = (1 - level) / 2
    if by:
        q = calib.groupby(by)["ratio"].quantile([a, 1 - a]).unstack()
        q.columns = ["lo", "hi"]
        d = check.merge(q.reset_index(), on=by, how="left")
    else:
        d = check.copy()
        d["lo"], d["hi"] = calib["ratio"].quantile(a), calib["ratio"].quantile(1 - a)
    d["lower"], d["upper"] = d["forecast"] * d["lo"], d["forecast"] * d["hi"]
    return d

rows = []
for name in ["pooled", "by warehouse"]:
    for lvl in LEVELS:
        d = interval(cal, chk, options[name], lvl)
        inside = ((d["actual"] >= d["lower"]) & (d["actual"] <= d["upper"])).mean()
        rows.append({"grouping": name, "asked for": lvl, "got (validation check)": round(inside, 3)})
pd.DataFrame(rows)

,grouping,asked for,got (validation check)
0,pooled,0.5,0.562
1,pooled,0.8,0.750
2,pooled,0.9,0.920
3,by warehouse,0.5,0.571
4,by warehouse,0.8,0.786
5,by warehouse,0.9,0.875


### How we got here (the process)
We repeated the check at three range sizes for the two surviving options, pooled and by warehouse.

### How to read it
"Asked for" is the share of days we wanted inside the range. "Got" is what the last 28 validation days did.

### What it means
Both track the request: pooled gave 0.56 / 0.75 / 0.92 for 50% / 80% / 90%, and by warehouse 0.57 / 0.79 / 0.88. Neither is clearly better. The pooled version is simpler and rests on four times as many data points per percentile.

### What's next and why
We choose **pooled** (`GROUPING = None`) and learn the ratios on **all** 56 validation days. Then we score the 56 test days once, which we have not looked at for any choice so far.

## 5d – Final ranges, scored once on the test days
*What we are looking for:* with ratios learned from **all** validation days, how often do the 56 test days land inside the range? The test days have not been used for any choice so far.

In [13]:
GROUPING = None   # set after we read 5b/5c together: None (pooled) or ["warehouse"]

final = {lvl: interval(val, test, GROUPING, lvl) for lvl in LEVELS}

rows = []
for lvl, d in final.items():
    inside = (d["actual"].between(d["lower"], d["upper"]))
    below = (d["actual"] < d["lower"]).mean()
    above = (d["actual"] > d["upper"]).mean()
    rows.append({"range": f"{int(lvl*100)}%",
                 "inside": round(inside.mean(), 3),
                 "below range (waste risk)": round(below, 3),
                 "above range (stockout risk)": round(above, 3),
                 "avg width (% of forecast)": round(((d["upper"]-d["lower"])/d["forecast"]).mean(), 3)})
pd.DataFrame(rows)

,range,inside,below range (waste risk),above range (stockout risk),avg width (% of forecast)
0,50%,0.625,0.210,0.165,0.095
1,80%,0.906,0.054,0.040,0.196
2,90%,0.969,0.009,0.022,0.239


### How we got here (the process)
Ratios from all validation days, applied to the model's test forecasts. Test forecasts come from the model refit on training plus validation. This is the one time the test days are used for the ranges.

### How to read it
Each row is one range size. "Inside" is the share of test days where real sales fell within it. "Below range" means sales were lower than the bottom (stock we would have wasted). "Above range" means sales were higher than the top (a stockout risk).

### What it means
- **80% range:** 90.6% of test days landed inside, with 5.4% below and 4.0% above. The range is a little too wide for this window, and the misses are balanced between the two sides.
- **90% range:** 96.9% inside. **50% range:** 62.5% inside.
- The ranges are cautious because the test weeks were easier to forecast (test WAPE 3.74%) than the validation weeks they were learned from (5.88%).
- The average width of the 80% range is 19.6% of the forecast, about ±10%.

### What's next and why
The ranges did not undercover, which is the safer way to be wrong. We will not retune them to the test days. Next we check each warehouse.

In [14]:
# Same check by warehouse for the 80% range
d80 = final[0.8].copy()
d80["inside"] = d80["actual"].between(d80["lower"], d80["upper"])
d80.groupby("warehouse")["inside"].mean().round(3)

warehouse
Cuenca           0.893
Guayaquil        0.929
Quito            0.964
Santo Domingo    0.839
Name: inside, dtype: float64

### How we got here (the process)
Same 80% range, split by warehouse, to see whether pooling is unfair to any one.

### How to read it
Each number is the share of that warehouse's 56 test days inside its 80% range. Each value rests on 56 days, so one day is worth 1.8 points.

### What it means
Quito 96.4%, Guayaquil 92.9%, Cuenca 89.3%, Santo Domingo 83.9%. All are at or above 80%, but the smaller warehouses have more misses. This matches the forecast error: Santo Domingo's WAPE is 5.72% and Quito's is 3.02%. A single pooled range is therefore tighter than ideal for the small warehouses and looser than needed for Quito.

### What's next and why
This is the main limit of the pooled choice. With about 2.6 years of history, per-warehouse ratios rest on only 56 days each, which is too thin to trust. We keep pooled now and revisit per-warehouse ranges if more history becomes available.

In [15]:
# Chart: one warehouse, test window, 80% and 50% ranges
wh = "Quito"
fig = go.Figure()
for lvl, color in [(0.8, "rgba(31,119,180,0.20)"), (0.5, "rgba(31,119,180,0.40)")]:
    d = final[lvl][final[lvl]["warehouse"] == wh].sort_values("date")
    fig.add_trace(go.Scatter(x=d["date"], y=d["upper"], mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=d["date"], y=d["lower"], mode="lines", line=dict(width=0), fill="tonexty",
                             fillcolor=color, name=f"{int(lvl*100)}% range"))
d = final[0.8][final[0.8]["warehouse"] == wh].sort_values("date")
fig.add_trace(go.Scatter(x=d["date"], y=d["forecast"], mode="lines", name="forecast", line=dict(color="#1f77b4")))
fig.add_trace(go.Scatter(x=d["date"], y=d["actual"], mode="markers", name="actual", marker=dict(color="black", size=6)))
fig.update_layout(title=f"{wh}: forecast range vs actual units, test window", yaxis_title="units per day", height=450)
fig.show()

### How to read it
Quito, test window. The blue line is the forecast, the shaded bands are the 50% and 80% ranges, and the black dots are what actually sold.

### What it means
Nearly every dot sits inside the 80% band. The weekly shape (a peak each weekend, a dip midweek) is followed by the bands. The few misses are at peaks, such as the high point around Jun 24, where sales ran above the top of the range.

### What's next and why
Peaks are where a stockout hurts most, so the stocking rule should look at the upper end of the range.

## 5e – Save the ranges for the stocking rule
*What we are looking for:* a file the next step reads.

In [16]:
out = final[0.8][["date", "warehouse", "actual", "forecast", "lower", "upper"]].rename(
    columns={"lower": "lower80", "upper": "upper80"})
out = out.merge(final[0.5][["date", "warehouse", "lower", "upper"]].rename(
    columns={"lower": "lower50", "upper": "upper50"}), on=["date", "warehouse"])
out.to_csv(PROCESSED / "forecast_ranges_test.csv", index=False)

# Also save the ratio percentiles the app will use on a brand-new forecast
q = val["ratio"].quantile([.05, .10, .25, .5, .75, .90, .95]).rename("ratio")
q.to_csv(PROCESSED / "ratio_percentiles.csv")
print("saved", len(out), "rows;", "ratio percentiles:", q.round(3).to_dict())

saved 224 rows; ratio percentiles: {0.05: 0.873, 0.1: 0.899, 0.25: 0.957, 0.5: 0.998, 0.75: 1.051, 0.9: 1.095, 0.95: 1.112}


### What it means
`forecast_ranges_test.csv` (test days with 50% and 80% ranges) and `ratio_percentiles.csv` (the percentiles the app will multiply a new forecast by) are saved.

### Assumptions and limits
- Ratios come from 56 validation days (224 rows), so the 5% and 95% percentiles are rough.
- The model used actual earlier days for its lags (one-day-ahead), so ranges for forecasts made several days ahead would be wider.
- The test window lacks Mother's Day, May 1 and year-end, so the ranges are untested for those days.
- Pooled ranges are tighter than ideal for the small warehouses (Santo Domingo, Cuenca).
- Sales are not demand: days that sold out look smaller than real demand.